In [76]:
import pandas as pd
import numpy as np

import torch
import torch.nn as nn

from torch.utils.data import (
    Dataset,
    DataLoader
)

In [77]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(device)

cuda


In [78]:
train_df = pd.read_csv(
    "../data/processed/train.csv"
)

test_df = pd.read_csv(
    "../data/processed/test.csv"
)

print(train_df.shape)
print(test_df.shape)

(121824, 5)
(30456, 5)


In [79]:
from torch.utils.data import Dataset


class BookDataset(Dataset):

    def __init__(self, dataframe):

        self.users = torch.tensor(
            dataframe["user_idx"].values,
            dtype=torch.long
        )

        self.books = torch.tensor(
            dataframe["book_idx"].values,
            dtype=torch.long
        )

        self.ratings = torch.tensor(
            dataframe["Rating"].values,
            dtype=torch.float32
        )

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, idx):

        return (
            self.users[idx],
            self.books[idx],
            self.ratings[idx]
        )

In [80]:
train_dataset = BookDataset(train_df)
test_dataset = BookDataset(test_df)

print(len(train_dataset))
print(len(test_dataset))

121824
30456


In [81]:
user, book, rating = train_dataset[0]

print(user)
print(book)
print(rating)

tensor(11120)
tensor(3250)
tensor(10.)


In [82]:
from torch.utils.data import DataLoader

BATCH_SIZE = 1024

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [83]:
users, books, ratings = next(
    iter(train_loader)
)

print(users.shape)
print(books.shape)
print(ratings.shape)

torch.Size([1024])
torch.Size([1024])
torch.Size([1024])


In [84]:
print(users[:5])
print(books[:5])
print(ratings[:5])

tensor([2114, 1370, 1889,  137, 6843])
tensor([2188, 2243, 2271, 1023,  945])
tensor([7., 8., 9., 8., 9.])


In [85]:
import torch.nn as nn


class MatrixFactorization(nn.Module):

    def __init__(
        self,
        n_users,
        n_books,
        embedding_dim=50
    ):
        super().__init__()

        self.user_embedding = nn.Embedding(
            n_users,
            embedding_dim
        )

        self.book_embedding = nn.Embedding(
            n_books,
            embedding_dim
        )

    def forward(
        self,
        user_ids,
        book_ids
    ):
        user_vecs = self.user_embedding(
            user_ids
        )

        book_vecs = self.book_embedding(
            book_ids
        )

        prediction = (
            user_vecs * book_vecs
        ).sum(dim=1)

        return prediction

In [86]:
n_users = (
    max(
        train_df["user_idx"].max(),
        test_df["user_idx"].max()
    )
    + 1
)

n_books = (
    max(
        train_df["book_idx"].max(),
        test_df["book_idx"].max()
    )
    + 1
)

print(n_users)
print(n_books)

13305
14513


In [87]:
model = MatrixFactorization(
    n_users=n_users,
    n_books=n_books,
    embedding_dim=50
).to(device)

predictions = model(
    users.to(device),
    books.to(device)
)

print(predictions.shape)
print(predictions[:10])

torch.Size([1024])
tensor([  6.7967,   4.8863, -13.9195,   3.5003,  -2.0889,  -8.8738,   1.5223,
          0.0178,   9.1099,   9.4485], device='cuda:0',
       grad_fn=<SliceBackward0>)


In [88]:
criterion = nn.MSELoss()

In [89]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [90]:
ratings = ratings.to(device)

loss = criterion(
    predictions,
    ratings
)

print(loss.item())

118.43350219726562


In [91]:
EPOCHS = 5

for epoch in range(EPOCHS):

    model.train()

    total_loss = 0

    for users, books, ratings in train_loader:

        users = users.to(device)
        books = books.to(device)
        ratings = ratings.to(device)

        optimizer.zero_grad()

        predictions = model(
            users,
            books
        )

        loss = criterion(
            predictions,
            ratings
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = (
        total_loss /
        len(train_loader)
    )

    print(
        f"Epoch {epoch+1}: "
        f"{avg_loss:.4f}"
    )

Epoch 1: 112.4552
Epoch 2: 106.3299
Epoch 3: 100.4322
Epoch 4: 94.8221
Epoch 5: 89.5242


In [92]:
import math

model.eval()

total_loss = 0

with torch.no_grad():

    for users, books, ratings in test_loader:

        users = users.to(device)
        books = books.to(device)
        ratings = ratings.to(device)

        predictions = model(
            users,
            books
        )

        loss = criterion(
            predictions,
            ratings
        )

        total_loss += loss.item()

avg_test_loss = (
    total_loss /
    len(test_loader)
)

rmse = math.sqrt(
    avg_test_loss
)

print("Test Loss:", avg_test_loss)
print("RMSE:", rmse)

Test Loss: 108.81370391845704
RMSE: 10.431380729244669


In [93]:
import torch
import torch.nn as nn


class MatrixFactorization(nn.Module):

    def __init__(
        self,
        n_users,
        n_books,
        global_mean,
        embedding_dim=50
    ):
        super().__init__()

        self.user_embedding = nn.Embedding(
            n_users,
            embedding_dim
        )

        self.book_embedding = nn.Embedding(
            n_books,
            embedding_dim
        )

        self.user_bias = nn.Embedding(
            n_users,
            1
        )

        self.book_bias = nn.Embedding(
            n_books,
            1
        )

        self.global_mean = global_mean

    def forward(
        self,
        user_ids,
        book_ids
    ):

        user_vecs = self.user_embedding(
            user_ids
        )

        book_vecs = self.book_embedding(
            book_ids
        )

        interaction = (
            user_vecs * book_vecs
        ).sum(dim=1)

        user_bias = self.user_bias(
            user_ids
        ).squeeze()

        book_bias = self.book_bias(
            book_ids
        ).squeeze()

        prediction = (
            self.global_mean
            + user_bias
            + book_bias
            + interaction
        )

        return prediction

In [94]:
global_mean = train_df["Rating"].mean()

print(global_mean)

7.771851195166798


In [95]:
model = MatrixFactorization(
    n_users=n_users,
    n_books=n_books,
    global_mean=global_mean,
    embedding_dim=50
).to(device)

In [96]:
criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

EPOCHS = 30

for epoch in range(EPOCHS):

    model.train()

    total_loss = 0

    for users, books, ratings in train_loader:

        users = users.to(device)
        books = books.to(device)
        ratings = ratings.to(device)

        optimizer.zero_grad()

        predictions = model(
            users,
            books
        )

        loss = criterion(
            predictions,
            ratings
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Loss: {avg_loss:.4f}"
    )

Epoch 1/30 | Loss: 54.8088
Epoch 2/30 | Loss: 49.9745
Epoch 3/30 | Loss: 45.4719
Epoch 4/30 | Loss: 41.3343
Epoch 5/30 | Loss: 37.5683
Epoch 6/30 | Loss: 34.1459
Epoch 7/30 | Loss: 31.0361
Epoch 8/30 | Loss: 28.2131
Epoch 9/30 | Loss: 25.6497
Epoch 10/30 | Loss: 23.3192
Epoch 11/30 | Loss: 21.2017
Epoch 12/30 | Loss: 19.2746
Epoch 13/30 | Loss: 17.5228
Epoch 14/30 | Loss: 15.9290
Epoch 15/30 | Loss: 14.4780
Epoch 16/30 | Loss: 13.1590
Epoch 17/30 | Loss: 11.9572
Epoch 18/30 | Loss: 10.8637
Epoch 19/30 | Loss: 9.8687
Epoch 20/30 | Loss: 8.9631
Epoch 21/30 | Loss: 8.1396
Epoch 22/30 | Loss: 7.3901
Epoch 23/30 | Loss: 6.7090
Epoch 24/30 | Loss: 6.0899
Epoch 25/30 | Loss: 5.5268
Epoch 26/30 | Loss: 5.0159
Epoch 27/30 | Loss: 4.5519
Epoch 28/30 | Loss: 4.1304
Epoch 29/30 | Loss: 3.7480
Epoch 30/30 | Loss: 3.4015


In [98]:
import math

model.eval()

total_loss = 0

with torch.no_grad():

    for users, books, ratings in test_loader:

        users = users.to(device)
        books = books.to(device)
        ratings = ratings.to(device)

        predictions = model(
            users,
            books
        )

        loss = criterion(
            predictions,
            ratings
        )

        total_loss += loss.item()

avg_test_loss = total_loss / len(test_loader)

rmse = math.sqrt(avg_test_loss)

print("Test Loss:", avg_test_loss)
print("RMSE:", rmse)

Test Loss: 39.223764928181964
RMSE: 6.262887906404039


In [99]:
book_embeddings = (
    model.book_embedding.weight
    .detach()
    .cpu()
    .numpy()
)

print(book_embeddings.shape)

(14513, 50)


In [100]:
book_embeddings[0][:10]

array([-0.30572912, -0.21946742, -0.4647533 ,  0.35641712, -0.16437592,
        0.81973726,  0.41518682,  1.3826164 , -1.8640232 ,  0.40583426],
      dtype=float32)

In [104]:
books_metadata = pd.read_csv(
    "../data/processed/books_metadata.csv"
)

isbn_to_title = dict(
    zip(
        books_metadata["ISBN"],
        books_metadata["Title"]
    )
)

In [106]:
from sklearn.metrics.pairwise import cosine_similarity

target_idx = 0

target_embedding = (
    book_embeddings[target_idx]
    .reshape(1, -1)
)

similarities = cosine_similarity(
    target_embedding,
    book_embeddings
)[0]

In [107]:
import numpy as np

top_indices = np.argsort(
    similarities
)[::-1][:10]

print(top_indices)

[    0  2399  3595  6332  3625 10137  6905  7614  6107  5197]


In [111]:
idx_to_book = dict(
    zip(
        train_df["book_idx"],
        train_df["ISBN"]
    )
)

print(idx_to_book[0])

0060517794


In [114]:
isbn_to_title = dict(
    zip(
        books_metadata["ISBN"],
        books_metadata["Title"]
    )
)

In [115]:
idx_to_book = dict(
    zip(
        train_df["book_idx"],
        train_df["ISBN"]
    )
)

In [112]:
for idx in top_indices:

    isbn = idx_to_book[idx]

    print(
        f"Similarity: {similarities[idx]:.4f}"
    )

    print(
        isbn_to_title.get(
            isbn,
            "Unknown"
        )
    )

    print("-" * 50)

Similarity: 1.0000
Little Altars Everywhere
--------------------------------------------------
Similarity: 0.5218
Unknown
--------------------------------------------------
Similarity: 0.4985
Longitude: The True Story of a Lone Genius Who Solved the Greatest Scientific Problem of His Time
--------------------------------------------------
Similarity: 0.4853
Gulliver's Travels (Penguin Popular Classics)
--------------------------------------------------
Similarity: 0.4647
First Things First: To Live, to Love, to Learn, to Leave a Legacy
--------------------------------------------------
Similarity: 0.4457
Sense and Sensibility (Wordsworth Classics)
--------------------------------------------------
Similarity: 0.4395
PERFUME
--------------------------------------------------
Similarity: 0.4378
The Devil in the White City: Murder, Magic, and Madness at the Fair that Changed America
--------------------------------------------------
Similarity: 0.4370
A Language Older Than Words
---------

In [117]:
book_stats = (
    train_df
    .groupby("ISBN")
    .agg(
        avg_rating=("Rating", "mean"),
        num_ratings=("Rating", "count")
    )
)

In [118]:
book_stats.sort_values(
    "num_ratings",
    ascending=False
).head(20)

,avg_rating,num_ratings
ISBN,,
0316666343,8.252149,349
0385504209,8.556338,284
0971880107,4.091324,219
0312195516,8.360000,200
0142001740,8.489362,188
059035342X,9.000000,170
0060928336,7.909091,165
0679781587,8.560976,164
0446672211,8.245283,159


In [119]:
popular_isbn = "0316666343"

print(
    isbn_to_title.get(
        popular_isbn,
        "Unknown"
    )
)

The Lovely Bones: A Novel


In [125]:
book_to_idx = dict(
    zip(
        train_df["ISBN"],
        train_df["book_idx"]
    )
)

idx_to_book = dict(
    zip(
        train_df["book_idx"],
        train_df["ISBN"]
    )
)

In [126]:
target_idx = book_to_idx[
    popular_isbn
]

print(target_idx)

231


In [127]:
target_embedding = (
    book_embeddings[target_idx]
    .reshape(1, -1)
)

similarities = cosine_similarity(
    target_embedding,
    book_embeddings
)[0]

In [128]:
top_indices = np.argsort(
    similarities
)[::-1][:15]

In [133]:
for idx in top_indices:

    if idx not in idx_to_book:
        continue

    isbn = idx_to_book[idx]

    print(
        f"Similarity: {similarities[idx]:.4f}"
    )

    print(
        isbn_to_title.get(
            isbn,
            "Unknown"
        )
    )

    print("-" * 60)

Similarity: 1.0000
The Lovely Bones: A Novel
------------------------------------------------------------
Similarity: 0.5282
The Client
------------------------------------------------------------
Similarity: 0.5228
A Palm for Mrs. Pollifax
------------------------------------------------------------
Similarity: 0.4635
War for the Oaks
------------------------------------------------------------
Similarity: 0.4618
Santaland Diaries & Seasons Greetings: 2 Plays
------------------------------------------------------------
Similarity: 0.4540
A Bend in the Road
------------------------------------------------------------
Similarity: 0.4481
Short Stories of Ernest Hemingway (A Scribner classic)
------------------------------------------------------------
Similarity: 0.4411
The Bachman Books: Four Early Novels by Stephen King : Rage, the Long Walk, Roadwork, the Running Man
------------------------------------------------------------
Similarity: 0.4407
The Giver
-----------------------------

In [134]:
print(len(idx_to_book))
print(book_embeddings.shape)

14472
(14513, 50)


In [140]:
from src.models.neural_cf import (
    NeuralCollaborativeFiltering
)

In [141]:
model = NeuralCollaborativeFiltering(
    n_users=n_users,
    n_books=n_books,
    embedding_dim=50
).to(device)

In [142]:
users, books, ratings = next(
    iter(train_loader)
)

preds = model(
    users.to(device),
    books.to(device)
)

print(preds.shape)
print(preds[:5])

torch.Size([1024])
tensor([0.1402, 0.0716, 0.0457, 0.0173, 0.1594], device='cuda:0',
       grad_fn=<SliceBackward0>)


In [143]:

total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("Total Parameters:", total_params)
print("Trainable Parameters:", trainable_params)

Total Parameters: 1399477
Trainable Parameters: 1399477


In [144]:
for name, param in model.named_parameters():

    print(
        f"{name:30}",
        param.shape,
        param.numel()
    )

user_embedding.weight          torch.Size([13305, 50]) 665250
book_embedding.weight          torch.Size([14513, 50]) 725650
mlp.0.weight                   torch.Size([64, 100]) 6400
mlp.0.bias                     torch.Size([64]) 64
mlp.2.weight                   torch.Size([32, 64]) 2048
mlp.2.bias                     torch.Size([32]) 32
mlp.4.weight                   torch.Size([1, 32]) 32
mlp.4.bias                     torch.Size([1]) 1


In [147]:
model = NeuralCollaborativeFiltering(
    n_users=n_users,
    n_books=n_books,
    embedding_dim=50
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

users, books, ratings = next(iter(train_loader))

preds = model(
    users.to(device),
    books.to(device)
)

print(preds[:5])

tensor([0.1962, 0.2728, 0.2659, 0.2856, 0.1511], device='cuda:0',
       grad_fn=<SliceBackward0>)


In [148]:
import torch.nn as nn
import torch.optim as optim


criterion = nn.MSELoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

EPOCHS = 10

for epoch in range(EPOCHS):

    model.train()

    total_loss = 0

    for users, books, ratings in train_loader:

        users = users.to(device)
        books = books.to(device)
        ratings = ratings.to(device)

        optimizer.zero_grad()

        predictions = model(
            users,
            books
        )

        loss = criterion(
            predictions,
            ratings
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = (
        total_loss /
        len(train_loader)
    )

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Loss: {avg_loss:.4f}"
    )

Epoch 1/10 | Loss: 19.3302
Epoch 2/10 | Loss: 3.4883
Epoch 3/10 | Loss: 3.1698
Epoch 4/10 | Loss: 2.9354
Epoch 5/10 | Loss: 2.7486
Epoch 6/10 | Loss: 2.5959
Epoch 7/10 | Loss: 2.4639
Epoch 8/10 | Loss: 2.3467
Epoch 9/10 | Loss: 2.2439
Epoch 10/10 | Loss: 2.1506


In [149]:
import math

model.eval()

total_loss = 0

with torch.no_grad():

    for users, books, ratings in test_loader:

        users = users.to(device)
        books = books.to(device)
        ratings = ratings.to(device)

        predictions = model(
            users,
            books
        )

        loss = criterion(
            predictions,
            ratings
        )

        total_loss += loss.item()

avg_test_loss = total_loss / len(test_loader)

rmse = math.sqrt(avg_test_loss)

print("Test Loss:", avg_test_loss)
print("RMSE:", rmse)

Test Loss: 2.7928325176239013
RMSE: 1.6711769857270957


In [150]:
users, books, ratings = next(iter(test_loader))

users = users.to(device)
books = books.to(device)

with torch.no_grad():
    preds = model(users, books)

print("Predictions:")
print(preds[:10])

print()

print("Actual Ratings:")
print(ratings[:10])

Predictions:
tensor([6.6736, 7.4790, 8.9409, 7.7368, 7.5533, 5.9959, 7.5279, 8.3762, 7.5645,
        7.6569], device='cuda:0')

Actual Ratings:
tensor([7., 7., 7., 8., 7., 8., 7., 5., 6., 9.])


In [151]:
train_rmse = math.sqrt(avg_loss)

print(train_rmse)

1.4664831939414664


In [152]:
print(preds[:10])
print(ratings[:10])

tensor([6.6736, 7.4790, 8.9409, 7.7368, 7.5533, 5.9959, 7.5279, 8.3762, 7.5645,
        7.6569], device='cuda:0')
tensor([7., 7., 7., 8., 7., 8., 7., 5., 6., 9.])


In [153]:
print(train_df.columns)
print(test_df.columns)

Index(['User-ID', 'ISBN', 'Rating', 'user_idx', 'book_idx'], dtype='str')
Index(['User-ID', 'ISBN', 'Rating', 'user_idx', 'book_idx'], dtype='str')


In [154]:
train_user_books = (
    train_df
    .groupby("user_idx")["book_idx"]
    .apply(set)
    .to_dict()
)

print(len(train_user_books))

13008


In [155]:
sample_user = list(
    train_user_books.keys()
)[0]

print(sample_user)

print(
    list(
        train_user_books[sample_user]
    )[:10]
)

0
[1, 2]


In [156]:
user_id = 0

seen_books = train_user_books[user_id]

print(len(seen_books))
print(list(seen_books))

2
[1, 2]


In [157]:
all_books = np.arange(n_books)

candidate_books = [
    book
    for book in all_books
    if book not in seen_books
]

print(len(candidate_books))

14511


In [158]:
user_tensor = torch.tensor(
    [user_id] * len(candidate_books)
).to(device)

book_tensor = torch.tensor(
    candidate_books
).to(device)

In [159]:
model.eval()

with torch.no_grad():

    scores = model(
        user_tensor,
        book_tensor
    )

In [160]:
print(scores.shape)

torch.Size([14511])


In [161]:
print(scores.shape)

torch.Size([14511])


In [162]:
len(candidate_books)
scores.shape
scores[:10]

tensor([7.5589, 9.0091, 8.5535, 8.8546, 8.7671, 8.5556, 8.7761, 8.6903, 8.0164,
        8.9122], device='cuda:0')

In [163]:
top_k = 10

top_indices = torch.topk(
    scores,
    k=top_k
).indices.cpu().numpy()

print(top_indices)

[10561  5881  6339 10492  6335 13216  9702 11912 13141 12587]


In [164]:
torch.topk(scores, k=2)

torch.return_types.topk(
values=tensor([10.8127, 10.5003], device='cuda:0'),
indices=tensor([10561,  5881], device='cuda:0'))

In [165]:
recommended_books = [
    candidate_books[idx]
    for idx in top_indices
]

print(recommended_books)

[np.int64(10563), np.int64(5883), np.int64(6341), np.int64(10494), np.int64(6337), np.int64(13218), np.int64(9704), np.int64(11914), np.int64(13143), np.int64(12589)]


In [166]:
top_indices = torch.topk(scores, k=10).indices.cpu().numpy()

recommended_books = [
    candidate_books[idx]
    for idx in top_indices
]

print(recommended_books)

[np.int64(10563), np.int64(5883), np.int64(6341), np.int64(10494), np.int64(6337), np.int64(13218), np.int64(9704), np.int64(11914), np.int64(13143), np.int64(12589)]


In [168]:
for book_idx in recommended_books:

    isbn = idx_to_book[book_idx]

    print(book_idx, isbn)

10563 0316150363
5883 0804112959
6341 0064430170
10494 0385311400
6337 0064400069
13218 0060953020
9704 0006374921
11914 0152023984
13143 1400032644
12589 0060968966


In [169]:
for book_idx in recommended_books:

    isbn = idx_to_book[book_idx]

    title = isbn_to_title.get(
        isbn,
        "Unknown"
    )

    print(title)

Dogs and Their Women
Grand Jury
Goodnight Moon
Drums of Autumn
The Long Winter (Little House)
Pilgrim at Tinker Creek
Unknown
The Little Prince
The Diary of an American Au Pair: A Novel
Linda Goodman's Love Signs : A New Approach to the Human Heart


In [170]:
user_id = 0

user_train = train_df[
    train_df["user_idx"] == user_id
]

user_test = test_df[
    test_df["user_idx"] == user_id
]

print("Train Books:")
print(user_train[["ISBN", "Rating"]])

print()

print("Test Books:")
print(user_test[["ISBN", "Rating"]])

Train Books:
             ISBN  Rating
52818  0671537458       9
77844  0679776818       8

Test Books:
             ISBN  Rating
14398  0060517794       9


In [171]:
for isbn in user_train["ISBN"]:

    print(
        isbn,
        isbn_to_title.get(
            isbn,
            "Unknown"
        )
    )

print("\nTest Book:\n")

for isbn in user_test["ISBN"]:

    print(
        isbn,
        isbn_to_title.get(
            isbn,
            "Unknown"
        )
    )

0671537458 Waiting to Exhale
0679776818 Birdsong: A Novel of Love and War

Test Book:

0060517794 Little Altars Everywhere


In [172]:
test_users = test_df["user_idx"].unique()[:100]

print(len(test_users))

100


In [173]:
test_users = test_df["user_idx"].unique()[:100]

print(len(test_users))

100


In [174]:
def hit_at_10(user_id, k=10):

    # Books user already saw
    seen_books = train_user_books[user_id]

    # User's test books
    user_test_books = test_df[
        test_df["user_idx"] == user_id
    ]["book_idx"].tolist()

    # Skip users with no test books
    if len(user_test_books) == 0:
        return None

    hidden_book = user_test_books[0]

    # Candidate books
    candidate_books = [
        book
        for book in range(n_books)
        if book not in seen_books
    ]

    user_tensor = torch.tensor(
        [user_id] * len(candidate_books),
        device=device
    )

    book_tensor = torch.tensor(
        candidate_books,
        device=device
    )

    model.eval()

    with torch.no_grad():

        scores = model(
            user_tensor,
            book_tensor
        )

    top_indices = torch.topk(
        scores,
        k=k
    ).indices.cpu().numpy()

    recommendations = [
        candidate_books[idx]
        for idx in top_indices
    ]

    return int(
        hidden_book in recommendations
    )

In [175]:
print(
    hit_at_10(0)
)

0


In [176]:
test_users = test_df[
    "user_idx"
].unique()[:100]

In [177]:
hits = []

for user_id in test_users:

    result = hit_at_10(user_id)

    if result is not None:
        hits.append(result)

print("Users Evaluated:", len(hits))
print("Hit@10:", sum(hits) / len(hits))

Users Evaluated: 100
Hit@10: 0.0


In [178]:
user_id = 0

seen_books = train_user_books[user_id]

hidden_book = (
    test_df[
        test_df["user_idx"] == user_id
    ]["book_idx"]
    .iloc[0]
)

candidate_books = [
    book
    for book in range(n_books)
    if book not in seen_books
]

user_tensor = torch.tensor(
    [user_id] * len(candidate_books),
    device=device
)

book_tensor = torch.tensor(
    candidate_books,
    device=device
)

with torch.no_grad():
    scores = model(
        user_tensor,
        book_tensor
    )

scores = scores.cpu().numpy()

In [179]:
book_to_score = dict(
    zip(
        candidate_books,
        scores
    )
)

sorted_books = sorted(
    book_to_score.items(),
    key=lambda x: x[1],
    reverse=True
)

rank = next(
    i + 1
    for i, (book, _) in enumerate(sorted_books)
    if book == hidden_book
)

print(rank)

14266


In [180]:
test_df.groupby(
    "user_idx"
).size().describe()

count    9025.000000
mean        3.374626
std         8.381252
min         1.000000
25%         1.000000
50%         2.000000
75%         3.000000
max       627.000000
dtype: float64

In [181]:
train_df.groupby(
    "user_idx"
).size().describe()

count    13008.000000
mean         9.365314
std         29.037281
min          1.000000
25%          3.000000
50%          5.000000
75%          9.000000
max       2655.000000
dtype: float64

In [182]:
popular_books = (
    train_df.groupby("book_idx")
    .size()
    .sort_values(ascending=False)
)

print(popular_books.head(20))

book_idx
231     349
41      284
86      219
145     200
755     188
707     170
945     165
343     164
347     159
464     158
349     156
234     151
420     142
1059    138
392     138
1640    137
991     137
872     135
1254    135
740     130
dtype: int64
